In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Step 1: Install and import required libraries
!pip install -q mediapipe opencv-python-headless

import mediapipe as mp
import cv2
import numpy as np
import os
from tqdm import tqdm




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 48.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 13.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<4.0.0dev,>=3.19.5, but you have protobuf 4.25.6 which is incompatible.
google-cloud-bigtable 2.27.0 requires google-api-core[grpc]<3.0.0dev,>=2.16.0, but you have google-api-core 1.34.1 which is incompatible.
pandas-gbq 0.25.0 requires google-api-core<3.0.0dev,>=2.10.2, but you have google-api-core 1.34.1 which is incompatible.
tensorflow-decision-forests 1.10.0 requires tensorflow==2.17.0, but you have tensorflow 2.17.1 which is incompatible.


In [141]:
# Step 2: Set dataset paths and define class labels
DATASET_PATH = "/kaggle/input/fer2013"
if not os.path.exists(DATASET_PATH):
    # Update this path if using a local or Colab environment
    DATASET_PATH = "path/to/your/fer2013/dataset"
train_dir = os.path.join(DATASET_PATH, "train")
val_dir = os.path.join(DATASET_PATH, "validation")
test_dir = os.path.join(DATASET_PATH, "test")

# Get class names from training directory and map to numeric labels
classes = sorted(os.listdir(train_dir))
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
print("Classes and label indices:", class_to_idx)


Classes and label indices: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


In [143]:
# Step 3: Extract face landmarks from training images and save features
X_train = []
y_train = []
with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5) as face_mesh:
    for cls_name, label in class_to_idx.items():
        class_folder = os.path.join(train_dir, cls_name)
        for img_name in tqdm(os.listdir(class_folder), desc=f"Processing {cls_name}", leave=False):
            img_path = os.path.join(class_folder, img_name)
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None:
                continue
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(image_rgb)
            if results.multi_face_landmarks:
                landmarks = results.multi_face_landmarks[0].landmark
                coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32).flatten()
                X_train.append(coords)
                y_train.append(label)
X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int64)
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
print("Train features shape:", X_train.shape, "Train labels shape:", y_train.shape)

Train features shape: (26530, 1404) Train labels shape: (26530,)


In [150]:
import numpy as np
from sklearn.model_selection import train_test_split

X_train_raw = np.load("X_train.npy")
y_train_raw = np.load("y_train.npy")

X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_train_raw
)

# 重新保存分割好的训练/验证数据
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_val.npy", X_val)
np.save("y_val.npy", y_val)



In [145]:
X_test = []
y_test = []
with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5) as face_mesh:
    for cls_name, label in class_to_idx.items():
        class_folder = os.path.join(test_dir, cls_name)
        if not os.path.exists(class_folder):
            continue  # skip if class not present in test set
        for img_name in tqdm(os.listdir(class_folder), desc=f"Processing {cls_name}", leave=False):
            img_path = os.path.join(class_folder, img_name)
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None:
                continue
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(image_rgb)
            if results.multi_face_landmarks:
                landmarks = results.multi_face_landmarks[0].landmark
                coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32).flatten()
                X_test.append(coords)
                y_test.append(label)
X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.int64)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)
print("Test features shape:", X_test.shape, "Test labels shape:", y_test.shape)

Test features shape: (6632, 1404) Test labels shape: (6632,)


In [151]:
# Step 6: Load the saved numpy feature arrays (if starting from here, skip if already in memory)
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_val = np.load("X_val.npy")
y_val = np.load("y_val.npy")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")
print("Loaded feature shapes:", X_train.shape, X_val.shape, X_test.shape)

Loaded feature shapes: (16979, 1404) (4245, 1404) (6632, 1404)


In [153]:
# 类别:    0    1    2    3    4    5    6
# 样本数: 100  200  80  500  150  700  50
counts = np.bincount(y_train)
print(counts)  # [100, 200, 80, 500, 150, 700, 50]
max_count = np.max(counts)  # 700
sampling_dict = {}
for cls_idx, c in enumerate(counts):
    desired = max(int(0.5 * max_count), c)  # 取 350 和原数量 c 中的更大者
    sampling_dict[cls_idx] = desired

print(sampling_dict)
# {0: 350, 1: 350, 2: 350, 3: 500, 4: 350, 5: 700, 6: 350}


[2247  241 2350 4428 3061 2736 1916]
{0: 2247, 1: 2214, 2: 2350, 3: 4428, 4: 3061, 5: 2736, 6: 2214}


In [154]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=sampling_dict,
    random_state=42
)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("过采样后类别分布:", np.bincount(y_train_res))



过采样后类别分布: [2247 2214 2350 4428 3061 2736 2214]


In [167]:
# Step 7: Build a lightweight MLP model for classification
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

input_dim = X_train_res.shape[1]  # number of features (468*3 = 1404)
model = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(7, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_59 (Dense)                     │ (None, 256)                 │         359,680 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_44 (Dropout)                 │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_60 (Dense)                     │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_45 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_61 (Dense)                     │ (None, 7)                   │             903 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 393,479 (1.50 MB)

 Trainable params: 393,479 (1.50 MB)

 Non-trainable params: 0 (0.00 B)

In [169]:
# Step 8: Train the MLP model with EarlyStopping and ModelCheckpoint callbacks
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)
history = model.fit(X_train_res, y_train_res, validation_data=(X_val, y_val),
                    epochs=1000, batch_size=64, callbacks=[early_stop, checkpoint], verbose=1)

Epoch 1/1000
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3508 - loss: 1.6535 - val_accuracy: 0.4137 - val_loss: 1.5695
Epoch 2/1000
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3539 - loss: 1.6486 - val_accuracy: 0.4122 - val_loss: 1.5567
Epoch 3/1000
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3551 - loss: 1.6515 - val_accuracy: 0.4014 - val_loss: 1.5763
Epoch 4/1000
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3521 - loss: 1.6517 - val_accuracy: 0.3962 - val_loss: 1.5765
Epoch 5/1000
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3518 - loss: 1.6540 - val_accuracy: 0.4271 - val_loss: 1.5295
Epoch 6/1000
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3517 - loss: 1.6550 - val_accuracy: 0.4224 - val_loss: 1.5345
Epoch 7/1000
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3617 - loss: 1.6417 - val_accuracy: 0.3901 - val_loss: 1.5906
Epoch 8/1000
301/301 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3624 - loss: 1.6336 - 

In [170]:
# Step 9: Save the final trained model for later use in real-time recognition
model.save("mlp10_model.h5")
print("Saved model to mlp10_model.h5")


Saved model to mlp10_model.h5


In [171]:
from tensorflow import keras

# 加载模型（如果使用自定义层需指定custom_objects）
model = keras.models.load_model("mlp10_model.h5")
model.summary()  # 验证模型结构是否正确加载

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_59 (Dense)                     │ (None, 256)                 │         359,680 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_44 (Dropout)                 │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_60 (Dense)                     │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_45 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_61 (Dense)                     │ (None, 7)                   │             903 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 393,481 (1.50 MB)

 Trainable params: 393,479 (1.50 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [172]:
import numpy as np

# 加载增强后的测试数据
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# 检查数据维度
print("测试集特征维度:", X_test.shape)
print("测试集标签维度:", y_test.shape)

测试集特征维度: (6632, 1404)
测试集标签维度: (6632,)


In [173]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"测试集准确率: {test_acc*100:.2f}%")

测试集准确率: 47.81%


In [162]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# 生成预测结果
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)  # 将概率转换为类别标签

# 分类报告（精确率/召回率/F1值）
print("\n分类报告:")
print(classification_report(y_test, y_pred_classes, target_names=classes))

# 混淆矩阵
print("\n混淆矩阵:")
print(confusion_matrix(y_test, y_pred_classes))

208/208 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

分类报告:
              precision    recall  f1-score   support

       angry       0.43      0.32      0.37       835
     disgust       0.10      0.69      0.17        97
        fear       0.29      0.11      0.16       933
       happy       0.80      0.79      0.80      1713
     neutral       0.42      0.53      0.47      1181
         sad       0.40      0.28      0.32      1094
    surprise       0.53      0.74      0.62       779

    accuracy                           0.50      6632
   macro avg       0.43      0.49      0.41      6632
weighted avg       0.51      0.50      0.49      6632


混淆矩阵:
[[ 267  153   26   65  152   89   83]
 [   8   67    0    1   14    3    4]
 [  86  117  100   70  233  136  191]
 [  47   79   26 1351   93   54   63]
 [  88  103   59   65  622  142  102]
 [  98  132  106   90  302  301   65]
 [  25   24   23   42   54   36  575]]
